In [ ]:
%env AWS_PROFILE=platform-developer

In [4]:
import os
import sys
from pathlib import Path

# `scripts` is a package but only pytest puts the project root on the path.
project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
if project_root not in sys.path:
    sys.path.append(project_root)

from adapters.extractors.oai_pmh.folio import config as folio_config
from adapters.extractors.oai_pmh.folio.enrichment.runtime import build_inventory_client
from adapters.extractors.oai_pmh.registry import get_config
from scripts.rebuild_adapter import (
    _build_download_client,
    _download_items_to_snapshot,
    _download_to_snapshot,
)
from utils.logger import ExecutionContext, get_trace_id, setup_logging

BIBS_PATH = Path("data/folio-raw-bibs.parquet")
ITEMS_PATH = Path("data/folio-raw-items.parquet")

config = get_config("folio")

In [ ]:
BIBS_PATH.parent.mkdir(parents=True, exist_ok=True)

# Runs for hours: ~1.7M records at 100 per OAI-PMH request. _download_to_snapshot
# writes chunks of 50,000 to `<path>.partial` and moves it into place only on
# success, so the guard below makes a re-run reuse a completed download rather
# than starting again.
if not BIBS_PATH.exists():
    with config.build_http_client() as http_client:
        oai_client = _build_download_client(config, http_client)
        _download_to_snapshot(oai_client, config.config, str(BIBS_PATH))

print(f"{BIBS_PATH} ({BIBS_PATH.stat().st_size / 1e9:.1f} GB)")

In [ ]:
# Items and holdings (with the UUIDs the OAI-PMH bib record cannot carry) come from
# mod-inventory-storage, keyed by the same store id as the bib rows. Reads the bib
# ids from the snapshot above, so that cell must have completed first.
if not ITEMS_PATH.exists():
    _download_items_to_snapshot(
        build_inventory_client(),
        bib_snapshot_path=str(BIBS_PATH),
        items_snapshot_path=str(ITEMS_PATH),
        namespace=folio_config.ITEMS_NAMESPACE,
    )

print(f"{ITEMS_PATH} ({ITEMS_PATH.stat().st_size / 1e9:.1f} GB)")

In [ ]:
import polars as pl

bibs = pl.scan_parquet(BIBS_PATH)
items = pl.scan_parquet(ITEMS_PATH)

print(f"{bibs.select(pl.len()).collect().item():,} bibs")
print(f"{items.select(pl.len()).collect().item():,} enriched instances")
bibs.head(3).collect()